In [1]:
#!wget https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4150nnn/GSM4150378/suppl/GSM4150378%5FsciPlex3%5FA549%5FMCF7%5FK562%5Fscreen%5Fgene.annotations.txt.gz

In [2]:
#!gzip -d GSM4150378_sciPlex3_A549_MCF7_K562_screen_gene.annotations.txt.gz

In [3]:
import pandas as pd
import anndata as ad
import numpy as np
import gzip
import re
import requests

In [4]:
df_genes = pd.read_csv('GSM4150378_sciPlex3_A549_MCF7_K562_screen_gene.annotations.txt', sep=' ')

In [5]:
df_genes = df_genes[~df_genes['id'].str.startswith('ENSMUS')]

In [6]:
df_genes['ens_id'] = df_genes['id'].str.split('_').str[0]

In [7]:
df_genes[df_genes['ens_id'].duplicated()]

,id,gene_short_name,ens_id
24,ENSG00000002586.18_PAR_Y,CD99,ENSG00000002586.18
5511,ENSG00000124333.15_PAR_Y,VAMP7,ENSG00000124333.15
5513,ENSG00000124334.17_PAR_Y,IL9R,ENSG00000124334.17
12090,ENSG00000167393.17_PAR_Y,PPP2R3B,ENSG00000167393.17
12483,ENSG00000168939.11_PAR_Y,SPRY3,ENSG00000168939.11
12518,ENSG00000169084.13_PAR_Y,DHRSX,ENSG00000169084.13
12522,ENSG00000169093.15_PAR_Y,ASMTL,ENSG00000169093.15
12524,ENSG00000169100.13_PAR_Y,SLC25A6,ENSG00000169100.13
14525,ENSG00000178605.13_PAR_Y,GTPBP6,ENSG00000178605.13
15189,ENSG00000182162.10_PAR_Y,P2RY8,ENSG00000182162.10


In [8]:
for item in df_genes[df_genes['ens_id'].duplicated(keep=False)][['id', 'ens_id']].values:
    if not (item[1] in item[0]):
        print(item)

In [9]:
target_versions = {}
for item in df_genes['id'].str.split('.').values:
    target_versions[item[0]] = item[1].split('_')[0]

In [10]:
rel = 90
url = f"https://ftp.ensembl.org/pub/release-{rel}/gtf/homo_sapiens/Homo_sapiens.GRCh38.{rel}.gtf.gz"
with requests.get(url, stream=True) as r:
    r.raise_for_status()
    handle = gzip.open(r.raw, 'rt')
    cnt = 0
    for line in handle:
        if line.startswith("#"):
            continue
        else:
            print(line)
            cnt += 1
        if cnt == 3:
            break

1	havana	gene	11869	14409	.	+	.	gene_id "ENSG00000223972"; gene_version "5"; gene_name "DDX11L1"; gene_source "havana"; gene_biotype "transcribed_unprocessed_pseudogene";

1	havana	transcript	11869	14409	.	+	.	gene_id "ENSG00000223972"; gene_version "5"; transcript_id "ENST00000456328"; transcript_version "2"; gene_name "DDX11L1"; gene_source "havana"; gene_biotype "transcribed_unprocessed_pseudogene"; transcript_name "DDX11L1-202"; transcript_source "havana"; transcript_biotype "processed_transcript"; tag "basic"; transcript_support_level "1";

1	havana	exon	11869	12227	.	+	.	gene_id "ENSG00000223972"; gene_version "5"; transcript_id "ENST00000456328"; transcript_version "2"; exon_number "1"; gene_name "DDX11L1"; gene_source "havana"; gene_biotype "transcribed_unprocessed_pseudogene"; transcript_name "DDX11L1-202"; transcript_source "havana"; transcript_biotype "processed_transcript"; exon_id "ENSE00002234944"; exon_version "1"; tag "basic"; transcript_support_level "1";



In [11]:
# Candidate Ensembl releases to check
#inspired by https://www.biostars.org/p/9486602/
releases = range(80, 113)  # adjust range as needed

def check_release(rel):
    url = f"https://ftp.ensembl.org/pub/release-{rel}/gtf/homo_sapiens/Homo_sapiens.GRCh38.{rel}.gtf.gz"
    try:
        with requests.get(url, stream=True) as r:
            r.raise_for_status()
            handle = gzip.open(r.raw, 'rt')
            found = {}
            for line in handle:
                if line.startswith("#"):
                    continue
                m = re.search(r'gene_id "([^"]+)"; gene_version "(\d+)";', line)
                if m:
                    gene, version = m.groups()
                    if gene in target_versions and gene not in found:
                        found[gene] = version
                        if len(found) == len(target_versions):
                            break
            handle.close()
            return found
    except Exception as e:
        print(f"Skipping {rel}: {e}")
        return {}

results = {}
for rel in releases:
    found = check_release(rel)
    if not found:
        continue
    match = sum(found.get(g) == v for g, v in target_versions.items())
    results[rel] = match
    print(f"Release {rel}: matched {match}/{len(target_versions)} genes")

best = max(results, key=results.get)
print(f"\nMost likely Ensembl release: {best}")

Release 80: matched 38067/58302 genes
Release 81: matched 53899/58302 genes
Release 82: matched 53899/58302 genes
Release 83: matched 54466/58302 genes
Release 84: matched 54466/58302 genes
Release 85: matched 56105/58302 genes
Release 86: matched 56105/58302 genes
Release 87: matched 56105/58302 genes
Release 88: matched 57626/58302 genes
Release 89: matched 57626/58302 genes
Release 90: matched 58302/58302 genes
Release 91: matched 58302/58302 genes
Release 92: matched 57006/58302 genes
Release 93: matched 57006/58302 genes
Release 94: matched 55678/58302 genes
Release 95: matched 55678/58302 genes
Release 96: matched 45204/58302 genes
Release 97: matched 42153/58302 genes
Release 98: matched 40176/58302 genes
Release 99: matched 39727/58302 genes
Release 100: matched 38913/58302 genes
Release 101: matched 38041/58302 genes
Release 102: matched 37439/58302 genes
Release 103: matched 36986/58302 genes
Release 104: matched 36169/58302 genes
Release 105: matched 34888/58302 genes
Releas

In [12]:
results = {}
for rel in releases:
    found = check_release(rel)
    if not found:
        continue
    match = sum(g in found.keys() for g, v in target_versions.items())
    results[rel] = match
    print(f"Release {rel}: matched {match}/{len(target_versions)} genes")

best = max(results, key=results.get)
print(f"\nMost likely Ensembl release: {best}")

Release 80: matched 57060/58302 genes
Release 81: matched 57175/58302 genes
Release 82: matched 57175/58302 genes
Release 83: matched 57304/58302 genes
Release 84: matched 57304/58302 genes
Release 85: matched 57704/58302 genes
Release 86: matched 57704/58302 genes
Release 87: matched 57704/58302 genes
Release 88: matched 58156/58302 genes
Release 89: matched 58156/58302 genes
Release 90: matched 58302/58302 genes
Release 91: matched 58302/58302 genes
Release 92: matched 58158/58302 genes
Release 93: matched 58158/58302 genes
Release 94: matched 58039/58302 genes
Release 95: matched 58039/58302 genes
Release 96: matched 57920/58302 genes
Release 97: matched 57852/58302 genes
Release 98: matched 57827/58302 genes
Release 99: matched 57814/58302 genes
Release 100: matched 57808/58302 genes
Release 101: matched 57771/58302 genes
Release 102: matched 57762/58302 genes
Release 103: matched 57738/58302 genes
Release 104: matched 57717/58302 genes
Release 105: matched 57687/58302 genes
Releas

In [13]:
adata_pert = ad.read_h5ad('./sciplex_perturbase/SrivatsanTrapnell2020_sciplex3.h5ad')

In [14]:
var_pert = adata_pert.var

In [15]:
var_pert = var_pert[~var_pert['ensembl_id'].str.startswith('ENSMUS')]

In [16]:
var_lamin = pd.read_parquet('./data/sciplex/raw/srivatsan20_sciplex3_var.parquet')

In [17]:
unique_genes = df_genes['id'].str.split('.').str[0].unique()
unique_genes_lamin = var_lamin.reset_index()['ensembl_id'].unique()
unique_genes_pert = var_pert['ensembl_id'].unique()

In [18]:
for i in range(len(unique_genes_lamin)):
    if unique_genes[i] != unique_genes_lamin[i]:
        print(i)

In [19]:
for i in range(len(unique_genes_pert)):
    if unique_genes[i] != unique_genes_pert[i]:
        print(i)